<a href="https://colab.research.google.com/github/ruicatzzz/aigc-detector/blob/daphne/notebooks/colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Set this to the repo root on your machine (the folder that contains src/).
# On Colab: clone the repo first, then cd into it.
%cd /path/to/aigc-detector

In [12]:
!pip install datasets --quiet


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import kagglehub
path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
print(path)

In [ ]:
import shutil, os

# path is whatever printed from the previous cell
os.makedirs('data/cifake', exist_ok=True)
shutil.copytree(path, 'data/cifake', dirs_exist_ok=True)

In [ ]:
# SID_Set subset (streamed from Hugging Face, no full ~140 GB download).
# Writes BOTH slices from one shared stream, so they never overlap:
#   data/sid_subset/{REAL,FAKE}        -> training  (first --n_train examples)
#   data/sid_test_holdout/{REAL,FAKE}  -> held-out  (next --n_test examples)
# SID_Set labels: 0 = real -> REAL; 1 = fully synthetic, 2 = tampered -> FAKE.
# Heads up: the streaming loader fetches a full shard before the first
# example, so the first printed line can take a few minutes.
!python -m src.download_sid --n_train 10000 --n_test 2000


In [ ]:
!python -m src.train --data_dir data/cifake/train data/sid_subset data/wildfake_subset --epochs 10 --out checkpoints/cnn_merged.pt

In [29]:
!python -m src.train --data_dir data/cifake/train data/sid_subset --epochs 10 --out checkpoints/cnn_merged.pt

Using device: cpu
data/cifake/train: 100000 images, classes={'FAKE': 0, 'REAL': 1}
data/sid_subset: 10000 images, classes={'FAKE': 0, 'REAL': 1}
Merged: 99000 train, 11000 val
Epoch 1/10: 100%|████████████████████████████| 774/774 [02:25<00:00,  5.31it/s]
Epoch 1: train_loss=0.4096  val_acc=0.8882  <- new best, saving
Epoch 2/10: 100%|████████████████████████████| 774/774 [03:12<00:00,  4.02it/s]
Epoch 2: train_loss=0.3162  val_acc=0.8940  <- new best, saving
Epoch 3/10: 100%|████████████████████████████| 774/774 [03:24<00:00,  3.79it/s]
Epoch 3: train_loss=0.2913  val_acc=0.9163  <- new best, saving
Epoch 4/10: 100%|████████████████████████████| 774/774 [03:34<00:00,  3.60it/s]
Epoch 4: train_loss=0.2713  val_acc=0.9166  <- new best, saving
Epoch 5/10: 100%|████████████████████████████| 774/774 [03:35<00:00,  3.59it/s]
Epoch 5: train_loss=0.2569  val_acc=0.9039
Epoch 6/10: 100%|████████████████████████████| 774/774 [03:22<00:00,  3.82it/s]
Epoch 6: train_loss=0.2418  val_acc=0.9247  <

In [ ]:
# output predictions to json file

!python -m src.infer --input_dir data/cifake/test data/sid_test_holdout data/wildfake_test_holdout --output_json outputs/preds.json --checkpoint checkpoints/cnn_merged.pt

In [ ]:
#robustness summary

!python -m src.robustness_summary --csv outputs/robustness_table.csv:CIFAKE+SID+WildFake

In [ ]:
# auc

!python -m src.eval_auc --checkpoint checkpoints/cnn_merged.pt --test_dir data/cifake/test data/sid_test_holdout data/wildfake_test_holdout --out_csv outputs/auc_table.csv

In [37]:
# auc

!python -m src.eval_auc --checkpoint checkpoints/cnn_merged.pt --test_dir data/cifake/test data/sid_test_holdout --out_csv outputs/auc_table.csv


=== REAL images ===

0233 (5).jpg:
  clean              pred=0.0174
  jpeg_q30           pred=0.0205
  jpeg_q70           pred=0.0176
  blur_sigma1.0      pred=0.0470
  blur_sigma2.0      pred=0.0682
  resize_0.5x        pred=0.0507
  resize_0.25x       pred=0.0437
  noise_sigma0.05    pred=0.0502
  color_jitter       pred=0.0497
  center_crop_80     pred=0.0248

0293 (8).jpg:
  clean              pred=0.0001
  jpeg_q30           pred=0.0003
  jpeg_q70           pred=0.0001
  blur_sigma1.0      pred=0.0025
  blur_sigma2.0      pred=0.0659
  resize_0.5x        pred=0.0021
  resize_0.25x       pred=0.0516
  noise_sigma0.05    pred=0.0007
  color_jitter       pred=0.0001
  center_crop_80     pred=0.0017

0854 (5).jpg:
  clean              pred=0.0001
  jpeg_q30           pred=0.0003
  jpeg_q70           pred=0.0001
  blur_sigma1.0      pred=0.0091
  blur_sigma2.0      pred=0.1133
  resize_0.5x        pred=0.0113
  resize_0.25x       pred=0.0594
  noise_sigma0.05    pred=0.0002
  color_ji